# Parte Comum: EDA do Obesity (estrutural, distribucional e outliers)

Dataset: Obesity (UCI 544), arquivo `data/parte_comum/obesity.csv`.

Pilares cobertos neste notebook: **estrutural**, **distribucional** e **outliers**.
O pilar relacional fica em `eda_relacional_comum.ipynb`.

In [1]:
# [IA] Claude Code – importacao das bibliotecas, leitura do dataset e pasta de saida
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

CAMINHO_DADOS = Path("../../data/parte_comum/obesity.csv")
PASTA_SAIDA = Path("../../out/parte_comum")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CAMINHO_DADOS)
COLUNA_CLASSE = "NObeyesdad"
df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [2]:
# [IA] Claude Code – ordem das classes (do menor para o maior nivel de peso), usada em todos os graficos
ORDEM_CLASSES = [
    "Insufficient_Weight", "Normal_Weight",
    "Overweight_Level_I", "Overweight_Level_II",
    "Obesity_Type_I", "Obesity_Type_II", "Obesity_Type_III",
]
assert set(df[COLUNA_CLASSE].unique()) == set(ORDEM_CLASSES)

## 1. Pilar estrutural

### 1.1 Dimensões

In [3]:
# [IA] Claude Code – quantidade de linhas e colunas
linhas, colunas = df.shape
print(f"Linhas: {linhas}")
print(f"Colunas: {colunas}")

Linhas: 2111
Colunas: 17


### 1.2 Tipos das variáveis

In [4]:
# [IA] Claude Code – tabela de tipos: dtype do pandas, quantidade de valores distintos e exemplos
tipos = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "qtd_valores_distintos": df.nunique(),
    "exemplos": [df[c].drop_duplicates().head(4).tolist() for c in df.columns],
})
tipos["grupo_pandas"] = ["numerica" if pd.api.types.is_numeric_dtype(df[c]) else "categorica" for c in df.columns]
tipos

,dtype,qtd_valores_distintos,exemplos,grupo_pandas
Gender,str,2,"[Female, Male]",categorica
Age,float64,1402,"[21.0, 23.0, 27.0, 22.0]",numerica
Height,float64,1574,"[1.62, 1.52, 1.8, 1.78]",numerica
Weight,float64,1525,"[64.0, 56.0, 77.0, 87.0]",numerica
family_history_with_overweight,str,2,"[yes, no]",categorica
FAVC,str,2,"[no, yes]",categorica
FCVC,float64,810,"[2.0, 3.0, 1.0, 2.450218]",numerica
NCP,float64,635,"[3.0, 1.0, 4.0, 3.28926]",numerica
CAEC,str,4,"[Sometimes, Frequently, Always, no]",categorica
SMOKE,str,2,"[no, yes]",categorica


In [5]:
# [IA] Claude Code – significado de cada coluna (dicionario da UCI) e classificacao estatistica
dicionario = pd.DataFrame([
    ("Gender", "Genero", "qualitativa nominal"),
    ("Age", "Idade (anos)", "quantitativa continua"),
    ("Height", "Altura (m)", "quantitativa continua"),
    ("Weight", "Peso (kg)", "quantitativa continua"),
    ("family_history_with_overweight", "Familiar com sobrepeso", "qualitativa nominal (binaria)"),
    ("FAVC", "Consome alimentos caloricos com frequencia", "qualitativa nominal (binaria)"),
    ("FCVC", "Frequencia de consumo de vegetais (1 a 3)", "quantitativa discreta / escala ordinal"),
    ("NCP", "Numero de refeicoes principais por dia (1 a 4)", "quantitativa discreta / escala ordinal"),
    ("CAEC", "Come entre as refeicoes", "qualitativa ordinal"),
    ("SMOKE", "Fuma", "qualitativa nominal (binaria)"),
    ("CH2O", "Consumo diario de agua (1 a 3)", "quantitativa discreta / escala ordinal"),
    ("SCC", "Monitora as calorias", "qualitativa nominal (binaria)"),
    ("FAF", "Frequencia de atividade fisica (0 a 3)", "quantitativa discreta / escala ordinal"),
    ("TUE", "Tempo usando aparelhos eletronicos (0 a 2)", "quantitativa discreta / escala ordinal"),
    ("CALC", "Consumo de alcool", "qualitativa ordinal"),
    ("MTRANS", "Meio de transporte", "qualitativa nominal"),
    ("NObeyesdad", "Nivel de obesidade (variavel de classe)", "qualitativa ordinal (alvo)"),
], columns=["coluna", "significado", "classificacao"]).set_index("coluna")
dicionario

,significado,classificacao
coluna,,
Gender,Genero,qualitativa nominal
Age,Idade (anos),quantitativa continua
Height,Altura (m),quantitativa continua
Weight,Peso (kg),quantitativa continua
family_history_with_overweight,Familiar com sobrepeso,qualitativa nominal (binaria)
FAVC,Consome alimentos caloricos com frequencia,qualitativa nominal (binaria)
FCVC,Frequencia de consumo de vegetais (1 a 3),quantitativa discreta / escala ordinal
NCP,Numero de refeicoes principais por dia (1 a 4),quantitativa discreta / escala ordinal
CAEC,Come entre as refeicoes,qualitativa ordinal


### Classificação das variáveis

A base possui 8 variáveis quantitativas e 9 qualitativas. `FCVC`, `NCP`, `CH2O`, `FAF` e `TUE` continuam contando como quantitativas, pois são medidas numéricas discretas, embora usem escalas ordinais. `CAEC` e `CALC` são qualitativas ordinais, e `NObeyesdad` é a variável de classe, também qualitativa ordinal, pois representa níveis crescentes de peso.

### 1.3 Qualidade dos dados

In [6]:
# [IA] Claude Code – valores ausentes (NaN) por coluna
ausentes = df.isna().sum()
print("Total de valores ausentes (NaN):", ausentes.sum())
ausentes[ausentes > 0]

Total de valores ausentes (NaN): 0


Series([], dtype: int64)

In [7]:
# [IA] Claude Code – linhas duplicadas (todas as colunas iguais)
duplicadas = df[df.duplicated(keep=False)]
print("Linhas duplicadas (copias extras):", df.duplicated().sum())
print("Linhas envolvidas em duplicacao (original + copias):", len(duplicadas))
duplicadas.sort_values(list(df.columns)).head(10)

Linhas duplicadas (copias extras): 24
Linhas envolvidas em duplicacao (original + copias): 33


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
295,Female,16.0,1.66,58.0,no,no,2.0,1.0,Sometimes,no,1.0,no,0.0,1.0,no,Walking,Normal_Weight
309,Female,16.0,1.66,58.0,no,no,2.0,1.0,Sometimes,no,1.0,no,0.0,1.0,no,Walking,Normal_Weight
282,Female,18.0,1.62,55.0,yes,yes,2.0,3.0,Frequently,no,1.0,no,1.0,1.0,no,Public_Transportation,Normal_Weight
460,Female,18.0,1.62,55.0,yes,yes,2.0,3.0,Frequently,no,1.0,no,1.0,1.0,no,Public_Transportation,Normal_Weight
97,Female,21.0,1.52,42.0,no,no,3.0,1.0,Frequently,no,1.0,no,0.0,0.0,Sometimes,Public_Transportation,Insufficient_Weight
98,Female,21.0,1.52,42.0,no,no,3.0,1.0,Frequently,no,1.0,no,0.0,0.0,Sometimes,Public_Transportation,Insufficient_Weight
523,Female,21.0,1.52,42.0,no,yes,3.0,1.0,Frequently,no,1.0,no,0.0,0.0,Sometimes,Public_Transportation,Insufficient_Weight
527,Female,21.0,1.52,42.0,no,yes,3.0,1.0,Frequently,no,1.0,no,0.0,0.0,Sometimes,Public_Transportation,Insufficient_Weight
659,Female,21.0,1.52,42.0,no,yes,3.0,1.0,Frequently,no,1.0,no,0.0,0.0,Sometimes,Public_Transportation,Insufficient_Weight
663,Female,21.0,1.52,42.0,no,yes,3.0,1.0,Frequently,no,1.0,no,0.0,0.0,Sometimes,Public_Transportation,Insufficient_Weight


In [8]:
# [IA] Claude Code – categorias de cada variavel categorica (procura de grafias inconsistentes)
colunas_categoricas = df.select_dtypes(exclude="number").columns
for coluna in colunas_categoricas:
    print(f"{coluna}: {df[coluna].value_counts().to_dict()}")

Gender: {'Male': 1068, 'Female': 1043}
family_history_with_overweight: {'yes': 1726, 'no': 385}
FAVC: {'yes': 1866, 'no': 245}
CAEC: {'Sometimes': 1765, 'Frequently': 242, 'Always': 53, 'no': 51}
SMOKE: {'no': 2067, 'yes': 44}
SCC: {'no': 2015, 'yes': 96}
CALC: {'Sometimes': 1401, 'no': 639, 'Frequently': 70, 'Always': 1}
MTRANS: {'Public_Transportation': 1580, 'Automobile': 457, 'Walking': 56, 'Motorbike': 11, 'Bike': 7}
NObeyesdad: {'Obesity_Type_I': 351, 'Obesity_Type_III': 324, 'Obesity_Type_II': 297, 'Overweight_Level_I': 290, 'Overweight_Level_II': 290, 'Normal_Weight': 287, 'Insufficient_Weight': 272}


In [9]:
# [IA] Claude Code – escalas que deveriam ser inteiras (questionario) mas tem valores fracionados
# A UCI informa que 77% dos dados foram gerados sinteticamente (SMOTE) e 23% vieram de questionario.
# O SMOTE interpola entre pontos, por isso aparecem valores como FCVC = 2.45.
ESCALAS = {"FCVC": (1, 3), "NCP": (1, 4), "CH2O": (1, 3), "FAF": (0, 3), "TUE": (0, 2)}

fracionados = pd.DataFrame({
    "faixa_esperada": [f"{a} a {b}" for a, b in ESCALAS.values()],
    "min": [df[c].min() for c in ESCALAS],
    "max": [df[c].max() for c in ESCALAS],
    "qtd_nao_inteiros": [(df[c] % 1 != 0).sum() for c in ESCALAS],
}, index=list(ESCALAS))
fracionados["pct_nao_inteiros"] = (fracionados["qtd_nao_inteiros"] / len(df) * 100).round(1)
fracionados

,faixa_esperada,min,max,qtd_nao_inteiros,pct_nao_inteiros
FCVC,1 a 3,1.0,3.0,826,39.1
NCP,1 a 4,1.0,4.0,640,30.3
CH2O,1 a 3,1.0,3.0,1290,61.1
FAF,0 a 3,0.0,3.0,1208,57.2
TUE,0 a 2,0.0,2.0,1153,54.6


In [10]:
# [IA] Claude Code – idade tambem aparece fracionada pelo mesmo motivo
print("Idades nao inteiras:", (df["Age"] % 1 != 0).sum(), f"({(df['Age'] % 1 != 0).mean() * 100:.1f}%)")

Idades nao inteiras: 1375 (65.1%)


In [11]:
# [IA] Claude Code – faixas das variaveis numericas (procura de valores impossiveis)
colunas_numericas = df.select_dtypes(include="number").columns.tolist()
faixas = df[colunas_numericas].describe().T[["min", "max", "mean", "std"]].round(3)
faixas["qtd_negativos"] = (df[colunas_numericas] < 0).sum()
faixas

,min,max,mean,std,qtd_negativos
Age,14.00,61.00,24.313,6.346,0
Height,1.45,1.98,1.702,0.093,0
Weight,39.00,173.00,86.586,26.191,0
FCVC,1.00,3.00,2.419,0.534,0
NCP,1.00,4.00,2.686,0.778,0
CH2O,1.00,3.00,2.008,0.613,0
FAF,0.00,3.00,1.010,0.851,0
TUE,0.00,2.00,0.658,0.609,0


### 1.4 Variável de classe

In [12]:
# [IA] Claude Code – contagem e proporcao de cada classe
classes = pd.DataFrame({
    "qtd": df[COLUNA_CLASSE].value_counts().reindex(ORDEM_CLASSES),
    "pct": (df[COLUNA_CLASSE].value_counts(normalize=True) * 100).reindex(ORDEM_CLASSES).round(2),
})
classes

,qtd,pct
NObeyesdad,,
Insufficient_Weight,272,12.88
Normal_Weight,287,13.60
Overweight_Level_I,290,13.74
Overweight_Level_II,290,13.74
Obesity_Type_I,351,16.63
Obesity_Type_II,297,14.07
Obesity_Type_III,324,15.35


In [13]:
# [IA] Claude Code – conferencia: a classe bate com o IMC (peso / altura^2) calculado?
# Faixas da OMS usadas pelos autores do dataset.
df_imc = df.assign(IMC=df["Weight"] / df["Height"] ** 2)
df_imc.groupby(COLUNA_CLASSE)["IMC"].describe().reindex(ORDEM_CLASSES).round(1)

,count,mean,std,min,25%,50%,75%,max
NObeyesdad,,,,,,,,
Insufficient_Weight,272.0,17.4,0.8,13.0,17.1,17.5,17.9,19.1
Normal_Weight,287.0,22.0,1.8,18.5,20.7,22.1,23.7,24.9
Overweight_Level_I,290.0,26.0,0.7,22.8,25.6,26.0,26.4,28.8
Overweight_Level_II,290.0,28.2,0.8,25.7,27.6,28.2,28.9,30.4
Obesity_Type_I,351.0,32.3,1.1,29.9,31.4,32.2,33.0,35.2
Obesity_Type_II,297.0,36.7,1.3,34.0,35.8,36.4,37.9,39.8
Obesity_Type_III,324.0,42.3,2.6,36.8,40.6,41.9,43.8,50.8


## Observações do pilar estrutural

Não há valores ausentes (NaN). Existem 24 linhas duplicadas extras, envolvendo 33 linhas quando se contam as cópias e seus registros originais. Os valores quebrados nas escalas inteiras aparecem porque 77% da base foi gerada por SMOTE, que interpola entre observações e produz valores como 2,45. As classes estão razoavelmente equilibradas, com proporções entre 12,9% e 16,6%. O IMC faz sentido em todas elas: as médias aumentam de 17,4 em `Insufficient_Weight` até 42,3 em `Obesity_Type_III`, acompanhando a ordem esperada das classes.

## 2. Pilar distribucional

### 2.1 Variáveis numéricas

In [14]:
# [IA] Claude Code – medidas resumo + assimetria e curtose de cada variavel numerica
resumo = df[colunas_numericas].describe().T
resumo["assimetria"] = df[colunas_numericas].skew()
resumo["curtose"] = df[colunas_numericas].kurt()
resumo.round(3)

,count,mean,std,min,25%,50%,75%,max,assimetria,curtose
Age,2111.0,24.313,6.346,14.00,19.947,22.778,26.000,61.00,1.529,2.826
Height,2111.0,1.702,0.093,1.45,1.630,1.700,1.768,1.98,-0.013,-0.563
Weight,2111.0,86.586,26.191,39.00,65.473,83.000,107.431,173.00,0.255,-0.700
FCVC,2111.0,2.419,0.534,1.00,2.000,2.386,3.000,3.00,-0.433,-0.638
NCP,2111.0,2.686,0.778,1.00,2.659,3.000,3.000,4.00,-1.107,0.386
CH2O,2111.0,2.008,0.613,1.00,1.585,2.000,2.477,3.00,-0.105,-0.879
FAF,2111.0,1.010,0.851,0.00,0.125,1.000,1.667,3.00,0.498,-0.621
TUE,2111.0,0.658,0.609,0.00,0.000,0.625,1.000,2.00,0.619,-0.549


In [15]:
# [IA] Claude Code – histogramas das variaveis numericas (com curva de densidade)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, coluna in zip(axes.flat, colunas_numericas):
    sns.histplot(df[coluna], bins=30, kde=True, ax=ax, color="#4C72B0")
    ax.axvline(df[coluna].mean(), color="#C44E52", linestyle="--", linewidth=1.5, label="media")
    ax.axvline(df[coluna].median(), color="#55A868", linestyle="-", linewidth=1.5, label="mediana")
    ax.set_title(dicionario.loc[coluna, "significado"])
    ax.set_xlabel(coluna)
    ax.set_ylabel("Quantidade de pessoas")
    ax.legend(fontsize=8)
fig.suptitle("Obesity: distribuicao das variaveis numericas", fontsize=15)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_histogramas_numericas.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\luisgx\AppData\Local\Temp\ipykernel_23052\3293461365.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Leitura das distribuições numéricas

`Age` tem a cauda mais longa, com a maior assimetria positiva (1,529); sua média (24,31) fica acima da mediana (22,78). As variáveis de questionário `FCVC`, `NCP`, `CH2O`, `FAF` e `TUE` apresentam picos associados aos valores inteiros de suas escalas, embora o SMOTE também gere valores fracionados. Em diferença absoluta, `Weight` tem a maior distância entre média e mediana (cerca de 3,6 kg), enquanto `Age` tem uma diferença de cerca de 1,5 ano. Como as unidades são diferentes, a comparação absoluta deve ser interpretada com cautela; em termos relativos à média, `NCP` apresenta a maior diferença, de aproximadamente 11,7%.

### 2.2 Variáveis categóricas

In [16]:
# [IA] Claude Code – frequencias absolutas e relativas de cada variavel categorica
for coluna in colunas_categoricas.drop(COLUNA_CLASSE):
    tabela = pd.DataFrame({
        "qtd": df[coluna].value_counts(),
        "pct": (df[coluna].value_counts(normalize=True) * 100).round(2),
    })
    print(f"\n{coluna} ({dicionario.loc[coluna, 'significado']})")
    print(tabela.to_string())


Gender (Genero)
         qtd    pct
Gender             
Male    1068  50.59
Female  1043  49.41

family_history_with_overweight (Familiar com sobrepeso)
                                 qtd    pct
family_history_with_overweight             
yes                             1726  81.76
no                               385  18.24

FAVC (Consome alimentos caloricos com frequencia)
       qtd    pct
FAVC             
yes   1866  88.39
no     245  11.61

CAEC (Come entre as refeicoes)
             qtd    pct
CAEC                   
Sometimes   1765  83.61
Frequently   242  11.46
Always        53   2.51
no            51   2.42

SMOKE (Fuma)
        qtd    pct
SMOKE             
no     2067  97.92
yes      44   2.08

SCC (Monitora as calorias)
      qtd    pct
SCC             
no   2015  95.45
yes    96   4.55

CALC (Consumo de alcool)
             qtd    pct
CALC                   
Sometimes   1401  66.37
no           639  30.27
Frequently    70   3.32
Always         1   0.05

MTRANS (Meio d

In [17]:
# [IA] Claude Code – graficos de barras das variaveis categoricas (barras horizontais ordenadas)
categoricas_sem_classe = colunas_categoricas.drop(COLUNA_CLASSE)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, coluna in zip(axes.flat, categoricas_sem_classe):
    contagem = df[coluna].value_counts().sort_values()
    ax.barh(contagem.index.astype(str), contagem.values, color="#4C72B0")
    for y, v in enumerate(contagem.values):
        ax.text(v, y, f" {v / len(df) * 100:.1f}%", va="center", fontsize=8)
    ax.set_title(dicionario.loc[coluna, "significado"])
    ax.set_xlabel("Quantidade de pessoas")
    ax.set_ylabel(coluna)
    ax.set_xlim(0, contagem.max() * 1.2)
fig.suptitle("Obesity: distribuicao das variaveis categoricas", fontsize=15)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_barras_categoricas.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\luisgx\AppData\Local\Temp\ipykernel_23052\1857579682.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# [IA] Claude Code – distribuicao da variavel de classe (na ordem do nivel de peso)
fig, ax = plt.subplots(figsize=(10, 5))
contagem = df[COLUNA_CLASSE].value_counts().reindex(ORDEM_CLASSES)
ax.bar(contagem.index, contagem.values, color="#4C72B0")
for x, v in enumerate(contagem.values):
    ax.text(x, v, f"{v}", ha="center", va="bottom", fontsize=9)
ax.set_title("Obesity: quantidade de pessoas em cada classe (NObeyesdad)")
ax.set_xlabel("Classe (nivel de peso)")
ax.set_ylabel("Quantidade de pessoas")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_distribuicao_classes.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\luisgx\AppData\Local\Temp\ipykernel_23052\3952933562.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Leitura das distribuições categóricas e da classe

As categorias que quase não aparecem são `CALC = Always` (0,05%), `MTRANS = Bike` (0,33%) e `MTRANS = Motorbike` (0,52%). Também são pouco frequentes `CAEC = Always` (2,51%) e `SMOKE = yes` (2,08%). `SMOKE` e `SCC` são variáveis binárias, mas quase todo mundo está na mesma categoria: 97,92% não fumam e 95,45% não monitoram as calorias. A distribuição das sete classes é razoavelmente equilibrada, sem uma classe dominante.

## 3. Outliers

Dois critérios: **IQR** (abaixo de Q1 − 1,5·IQR ou acima de Q3 + 1,5·IQR) e **z-score** (|z| > 3).
As escalas de questionário (FCVC, NCP, CH2O, FAF, TUE) têm faixa fechada, então um valor "extremo"
nelas é uma resposta válida da escala, não um erro.

In [19]:
# [IA] Claude Code – contagem de outliers por variavel pelos dois criterios
def limites_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

linhas_out = []
for coluna in colunas_numericas:
    inf, sup = limites_iqr(df[coluna])
    z = (df[coluna] - df[coluna].mean()) / df[coluna].std()
    linhas_out.append({
        "variavel": coluna,
        "limite_inferior_iqr": round(inf, 2),
        "limite_superior_iqr": round(sup, 2),
        "qtd_outliers_iqr": int(((df[coluna] < inf) | (df[coluna] > sup)).sum()),
        "qtd_outliers_z3": int((z.abs() > 3).sum()),
    })
outliers = pd.DataFrame(linhas_out).set_index("variavel")
outliers["pct_outliers_iqr"] = (outliers["qtd_outliers_iqr"] / len(df) * 100).round(2)
outliers

,limite_inferior_iqr,limite_superior_iqr,qtd_outliers_iqr,qtd_outliers_z3,pct_outliers_iqr
variavel,,,,,
Age,10.87,35.08,168,24,7.96
Height,1.42,1.98,1,0,0.05
Weight,2.54,170.37,1,1,0.05
FCVC,0.50,4.50,0,0,0.00
NCP,2.15,3.51,579,0,27.43
CH2O,0.25,3.82,0,0,0.00
FAF,-2.19,3.98,0,0,0.00
TUE,-1.50,2.50,0,0,0.00


In [20]:
# [IA] Claude Code – boxplots das variaveis numericas (cada uma na sua escala)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, coluna in zip(axes.flat, colunas_numericas):
    sns.boxplot(y=df[coluna], ax=ax, color="#8DA0CB", flierprops={"marker": "o", "markersize": 3})
    ax.set_title(f"{coluna}: {outliers.loc[coluna, 'qtd_outliers_iqr']} outliers (IQR)")
    ax.set_xlabel("")
    ax.set_ylabel(coluna)
fig.suptitle("Obesity: boxplots das variaveis numericas", fontsize=15)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_boxplots_numericas.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\luisgx\AppData\Local\Temp\ipykernel_23052\3268964925.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
# [IA] Claude Code – boxplots de Age, Height e Weight separados por classe
# Um valor que e outlier no geral pode ser normal dentro da propria classe (ex.: peso alto em Obesity_Type_III).
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, coluna in zip(axes, ["Age", "Height", "Weight"]):
    sns.boxplot(data=df, x=COLUNA_CLASSE, y=coluna, order=ORDEM_CLASSES, ax=ax, color="#8DA0CB",
                flierprops={"marker": "o", "markersize": 3})
    ax.set_title(f"{coluna} por classe")
    ax.set_xlabel("Classe (nivel de peso)")
    ax.set_ylabel(coluna)
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_boxplots_por_classe.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\luisgx\AppData\Local\Temp\ipykernel_23052\1355949317.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
# [IA] Claude Code – quem sao os outliers de Age, Height e Weight (para avaliar se sao plausiveis)
for coluna in ["Age", "Height", "Weight"]:
    inf, sup = limites_iqr(df[coluna])
    fora = df[(df[coluna] < inf) | (df[coluna] > sup)]
    print(f"\n{coluna}: {len(fora)} outliers | faixa dos outliers: {fora[coluna].min():.2f} a {fora[coluna].max():.2f}")
    print(fora[COLUNA_CLASSE].value_counts().to_string())


Age: 168 outliers | faixa dos outliers: 35.13 a 61.00
NObeyesdad
Obesity_Type_I         70
Overweight_Level_II    39
Obesity_Type_II        29
Overweight_Level_I     22
Normal_Weight           7
Insufficient_Weight     1

Height: 1 outliers | faixa dos outliers: 1.98 a 1.98
NObeyesdad
Obesity_Type_I    1

Weight: 1 outliers | faixa dos outliers: 173.00 a 173.00
NObeyesdad
Obesity_Type_III    1


### Discussão dos outliers

Os outliers parecem plausíveis na vida real, e não erros de digitação: idade entre 35,13 e 61 anos, altura de 1,98 m e peso de 173 kg são valores possíveis. Os 168 outliers de `Age` se concentram principalmente em `Obesity_Type_I` (70), `Overweight_Level_II` (39) e `Obesity_Type_II` (29). `NCP` também merece atenção: tem 579 outliers, o maior número da tabela, mas seus valores ficam dentro da escala válida de 1 a 4. Para essa variável, 2,15 e 3,51 são os limites inferior e superior do critério IQR; os 50% centrais ficam entre aproximadamente 2,66 e 3,00. Portanto, o critério IQR identifica valores extremos em relação à amostra, mas não prova que sejam inválidos.

In [23]:
# [IA] Claude Code – aplicar o tratamento escolhido pelo grupo
# Opcoes:
#   "manter":      nenhum valor e alterado (outliers considerados valores reais)
#   "winsorizar":  valores fora dos limites do IQR sao trocados pelo proprio limite
#   "remover":     linhas com outlier (IQR) em VARIAVEIS_TRATADAS sao removidas
TRATAMENTO = "manter"  # (decisao do grupo)
VARIAVEIS_TRATADAS = ["Age", "Height", "Weight", "NCP"]  # (decisao do grupo)

df_tratado = df.copy()
if TRATAMENTO == "manter":
    pass
elif TRATAMENTO == "winsorizar":
    for coluna in VARIAVEIS_TRATADAS:
        inf, sup = limites_iqr(df[coluna])
        df_tratado[coluna] = df_tratado[coluna].clip(inf, sup)
elif TRATAMENTO == "remover":
    mascara = pd.Series(False, index=df.index)
    for coluna in VARIAVEIS_TRATADAS:
        inf, sup = limites_iqr(df[coluna])
        mascara |= (df[coluna] < inf) | (df[coluna] > sup)
    df_tratado = df_tratado[~mascara]
else:
    raise ValueError(f"Tratamento desconhecido: {TRATAMENTO}")

print(f"Tratamento: {TRATAMENTO} | linhas antes: {len(df)} | linhas depois: {len(df_tratado)}")

Tratamento: manter | linhas antes: 2111 | linhas depois: 2111


### Justificativa do tratamento

O grupo decidiu manter os valores. Os outliers são plausíveis e podem representar pessoas reais, especialmente porque idade, altura e peso possuem ampla variação na população. Removê-los reduziria a amostra e poderia eliminar justamente perfis relevantes; winsorizar também distorceria os extremos. Em `NCP`, os 579 outliers do critério IQR continuam dentro da escala válida de 1 a 4, assim como os valores extremos das demais escalas de questionário, portanto são respostas válidas.

In [24]:
# [IA] Claude Code – salvar o dataset apos o tratamento de outliers (usado nas proximas etapas)
df_tratado.to_csv(PASTA_SAIDA / "obesity_tratado.csv", index=False)
print("Salvo em", PASTA_SAIDA / "obesity_tratado.csv")

Salvo em ..\..\out\parte_comum\obesity_tratado.csv
